In [1]:
import os
import json
import pandas as pd
from collections import Counter

# 1. SETUP PATH FOLDER
PROCESSED_CSV = os.path.abspath("data/processed/cases.csv")
QUERIES_JSON = os.path.abspath("data/eval/queries.json")
RESULTS_FOLDER = os.path.abspath("data/results")
os.makedirs(RESULTS_FOLDER, exist_ok=True)

# Load dataset utama dan kueri evaluasi
df = pd.read_csv(PROCESSED_CSV)
with open(QUERIES_JSON, 'r', encoding='utf-8') as f:
    queries_data = json.load(f)

# Jalankan kembali inisialisasi TF-IDF (pake pembagian data yang sama persis seperti Tahap 3)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['pasal'])
vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(df_train['ringkasan_fakta'])

# Panggil fungsi retrieve dari Tahap 3
def retrieve_internal(query: str, k: int = 5):
    query_vector = vectorizer.transform([query.lower()])
    similarities = cosine_similarity(query_vector, train_vectors).flatten()
    top_k_indices = similarities.argsort()[::-1][:k]
    
    top_k_cases = df_train.iloc[top_k_indices]
    return top_k_cases, similarities[top_k_indices]

# =====================================================================
# 2. IMPLEMENTASI FUNGSI PREDICT OUTCOME (Sesuai Panduan)
# =====================================================================
def predict_outcome(query: str, k: int = 5) -> dict:
    """
    Fungsi untuk memprediksi solusi kasus baru menggunakan Majority Vote 
    berdasarkan top-k kasus lama yang paling mirip.
    """
    # 1) Ambil top-k kasus lama yang paling mirip
    top_k_cases, similarity_scores = retrieve_internal(query, k=k)
    
    # 2) Ekstrak elemen solusi (Pasal Hukum) dari kasus-kasus tersebut
    solutions = top_k_cases['pasal'].tolist()
    case_ids = top_k_cases['case_id'].tolist()
    amar_list = top_k_cases['amar_putusan'].tolist()
    
    # 3) Terapkan Voting: Pilih solusi/pasal yang paling banyak muncul (Majority Vote)
    vote_counts = Counter(solutions)
    predicted_solution = vote_counts.most_common(1)[0][0]
    
    # Ambil contoh rekomendasi amar putusan dari kasus peringkat ke-1 (paling mirip)
    recommended_amar = amar_list[0]
    
    return {
        "top_k_case_ids": [int(cid) for cid in case_ids],
        "predicted_solution": predicted_solution,
        "recommended_amar": recommended_amar
    }

# =====================================================================
# 3. RUN PREDICTION PIPELINE & SIMPAN KE CSV
# =====================================================================
print("Memulai proses prediksi solusi untuk data pengujian...")

prediction_results = []

# Jalankan prediksi untuk setiap query yang ada di file evaluasi
for q in queries_data:
    prediction = predict_outcome(q['query_text'], k=5)
    
    prediction_results.append({
        "query_id": q['query_id'],
        "top_5_case_ids": str(prediction['top_k_case_ids']), # disimpan sebagai string list
        "predicted_solution": prediction['predicted_solution']
    })

# Ubah ke DataFrame dan simpan ke file csv hasil akhir sesuai instruksi tugas
df_predictions = pd.DataFrame(prediction_results)
predictions_csv_path = os.path.join(RESULTS_FOLDER, "predictions.csv")
df_predictions.to_csv(predictions_csv_path, index=False)

print(f"File hasil prediksi berhasil disimpan di: {predictions_csv_path}")

# =====================================================================
# 4. DEMO MANUAL (Bandingkan Prediksi vs Kenyataan)
# =====================================================================
print("\n--- DEMO MANUAL PERBANDINGAN SOLUSI ---")
demo_query = queries_data[0]['query_text']
demo_real_pasal = queries_data[0]['true_pasal']

res_demo = predict_outcome(demo_query, k=5)
print(f"Teks Kasus Baru (Potongan): {demo_query[:150]}...")
print(f"Top 5 Kasus Termirip (ID): {res_demo['top_k_case_ids']}")
print(f"-> Prediksi Solusi Sistem : {res_demo['predicted_solution']}")
print(f"-> Solusi Sebenarnya (Real): {demo_real_pasal}")

Memulai proses prediksi solusi untuk data pengujian...
File hasil prediksi berhasil disimpan di: d:\Kuliah\Semester 6\Computer Reasoning\Sub-CPMK3\cbr-pembunuhan\notebooks\data\results\predictions.csv

--- DEMO MANUAL PERBANDINGAN SOLUSI ---
Teks Kasus Baru (Potongan): mahkamah agung republik indonesia mahkamah agung republik indonesia mahkamah agung republik indonesia mahkamah agung republik indonesia mahkamah agung...
Top 5 Kasus Termirip (ID): [26, 29, 8, 33, 2]
-> Prediksi Solusi Sistem : Pasal 340 KUHP (Pembunuhan Berencana)
-> Solusi Sebenarnya (Real): Pasal 338 KUHP (Pembunuhan Biasa)
